# Door Condition Monitoring
## Feature Engineering

In [3]:
import pandas as pd
import numpy as np

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)


DATA_DIR = Path("../PS3/02_Datasets/Door")

TRAIN_PATH = DATA_DIR / "Train.csv"
SEGMENTS_PATH = Path("segmented_training_cycles.csv")

print("Train exists:", TRAIN_PATH.exists())
print("Segments exist:", SEGMENTS_PATH.exists())

train = pd.read_csv(TRAIN_PATH)
segments = pd.read_csv(SEGMENTS_PATH)

print("Train shape:", train.shape)
print("Segments shape:", segments.shape)

display(segments.head())

Train exists: True
Segments exist: True
Train shape: (18036, 17)
Segments shape: (110, 9)


,start_idx,end_idx,n_rows,start_timestamp,end_timestamp,segment_id,operation,status,duration_seconds
0,0,185,186,2023-07-05 00:00:00.000,2023-07-05 00:00:03.700,train_seg_001,Close,Normal,3.70
1,186,328,143,2023-07-05 00:00:23.999,2023-07-05 00:00:26.839,train_seg_002,Open,Normal,2.84
2,329,465,137,2023-07-05 00:00:51.266,2023-07-05 00:00:53.986,train_seg_003,Open,Abnormal resistance,2.72
3,466,652,187,2023-07-05 00:01:33.989,2023-07-05 00:01:37.709,train_seg_004,Close,Abnormal resistance,3.72
4,653,838,186,2023-07-05 00:02:24.608,2023-07-05 00:02:28.308,train_seg_005,Close,Abnormal resistance,3.70


In [4]:
# Convert the Door timestamp format into Pandas timestamps for cycle-level calculations.


def parse_door_datetime(value):
    parts = str(value).split("-")

    if len(parts) != 7:
        return pd.NaT

    try:
        year, month, day, hour, minute, second, ms = map(int, parts)

        return pd.Timestamp(year=year, month=month, day=day, hour=hour, minute=minute, second=second, microsecond=ms * 1000)

    except (ValueError, TypeError):
        return pd.NaT


train["timestamp"] = train["Datetime"].apply(parse_door_datetime)

In [5]:
# Extract one complete door operation from the original training stream.


def extract_cycle(train_df, segment_row):
    start_idx = int(segment_row["start_idx"])
    end_idx = int(segment_row["end_idx"])

    return train_df.loc[start_idx:end_idx].copy()

In [6]:
SIGNALS = ["Motor current(mA)", "Motor Voltage(10mV)", "Motor electrodynamic force", "Door leaf position"]

for signal in SIGNALS:
    print(signal, "->", signal in train.columns)

Motor current(mA) -> True
Motor Voltage(10mV) -> True
Motor electrodynamic force -> True
Door leaf position -> True


## Whole-Cycle Features

These features describe the overall behaviour of each signal during the
entire door operation.

They provide a simple baseline representation before we add temporal
information.

In [7]:
# Calculate basic statistics that summarise an entire signal trajectory.


def basic_statistics(values, prefix):

    values = np.asarray(values, dtype=float)

    return {
        f"{prefix}_mean": np.mean(values),
        f"{prefix}_std": np.std(values),
        f"{prefix}_min": np.min(values),
        f"{prefix}_max": np.max(values),
        f"{prefix}_range": (np.max(values) - np.min(values)),
    }

In [8]:
# Test the whole-cycle statistics on the first labelled operation.

example_cycle = extract_cycle(train, segments.iloc[0])

example_features = {}

for signal in SIGNALS:
    example_features.update(basic_statistics(example_cycle[signal], signal))

display(pd.DataFrame([example_features]))

,Motor current(mA)_mean,Motor current(mA)_std,Motor current(mA)_min,Motor current(mA)_max,Motor current(mA)_range,Motor Voltage(10mV)_mean,Motor Voltage(10mV)_std,Motor Voltage(10mV)_min,Motor Voltage(10mV)_max,Motor Voltage(10mV)_range,Motor electrodynamic force_mean,Motor electrodynamic force_std,Motor electrodynamic force_min,Motor electrodynamic force_max,Motor electrodynamic force_range,Door leaf position_mean,Door leaf position_std,Door leaf position_min,Door leaf position_max,Door leaf position_range
0,430.698925,523.670601,0.0,2208.0,2208.0,4146.236559,1416.459249,400.0,6200.0,5800.0,892.069892,444.411451,0.0,1331.0,1331.0,293.032258,235.365442,0.0,700.0,700.0


## Phase-Based Features

Each cycle is divided into five relative phases:

- 0–20%
- 20–40%
- 40–60%
- 60–80%
- 80–100%

For every signal and every phase, we calculate basic statistics.

This preserves coarse information about when a particular signal behaviour
occurs during the door operation.

In [10]:
# Define five relative phases covering the entire door operation.

PHASES = [(0.0, 0.2), (0.2, 0.4), (0.4, 0.6), (0.6, 0.8), (0.8, 1.0)]

In [11]:
# Calculate statistics for one signal within one relative phase of a cycle.


def phase_statistics(cycle, signal, phase_start, phase_end, phase_number):

    n = len(cycle)

    start_idx = int(np.floor(phase_start * n))

    end_idx = int(np.ceil(phase_end * n))

    values = pd.to_numeric(cycle[signal].iloc[start_idx:end_idx], errors="coerce").dropna().to_numpy()

    prefix = f"{signal}_phase{phase_number}"

    if len(values) == 0:
        return {f"{prefix}_mean": np.nan, f"{prefix}_std": np.nan, f"{prefix}_min": np.nan, f"{prefix}_max": np.nan, f"{prefix}_range": np.nan}

    return basic_statistics(values, prefix)

In [12]:
# Test the phase-based feature calculation on the example cycle.

example_phase_features = {}

for signal in SIGNALS:
    for phase_number, (phase_start, phase_end) in enumerate(PHASES, start=1):
        example_phase_features.update(phase_statistics(example_cycle, signal, phase_start, phase_end, phase_number))

display(pd.DataFrame([example_phase_features]))

,Motor current(mA)_phase1_mean,Motor current(mA)_phase1_std,Motor current(mA)_phase1_min,Motor current(mA)_phase1_max,Motor current(mA)_phase1_range,Motor current(mA)_phase2_mean,Motor current(mA)_phase2_std,Motor current(mA)_phase2_min,Motor current(mA)_phase2_max,Motor current(mA)_phase2_range,Motor current(mA)_phase3_mean,Motor current(mA)_phase3_std,Motor current(mA)_phase3_min,Motor current(mA)_phase3_max,Motor current(mA)_phase3_range,Motor current(mA)_phase4_mean,Motor current(mA)_phase4_std,Motor current(mA)_phase4_min,Motor current(mA)_phase4_max,Motor current(mA)_phase4_range,Motor current(mA)_phase5_mean,Motor current(mA)_phase5_std,Motor current(mA)_phase5_min,Motor current(mA)_phase5_max,Motor current(mA)_phase5_range,Motor Voltage(10mV)_phase1_mean,Motor Voltage(10mV)_phase1_std,Motor Voltage(10mV)_phase1_min,Motor Voltage(10mV)_phase1_max,Motor Voltage(10mV)_phase1_range,Motor Voltage(10mV)_phase2_mean,Motor Voltage(10mV)_phase2_std,Motor Voltage(10mV)_phase2_min,Motor Voltage(10mV)_phase2_max,Motor Voltage(10mV)_phase2_range,Motor Voltage(10mV)_phase3_mean,Motor Voltage(10mV)_phase3_std,Motor Voltage(10mV)_phase3_min,Motor Voltage(10mV)_phase3_max,Motor Voltage(10mV)_phase3_range,Motor Voltage(10mV)_phase4_mean,Motor Voltage(10mV)_phase4_std,Motor Voltage(10mV)_phase4_min,Motor Voltage(10mV)_phase4_max,Motor Voltage(10mV)_phase4_range,Motor Voltage(10mV)_phase5_mean,Motor Voltage(10mV)_phase5_std,Motor Voltage(10mV)_phase5_min,Motor Voltage(10mV)_phase5_max,Motor Voltage(10mV)_phase5_range,Motor electrodynamic force_phase1_mean,Motor electrodynamic force_phase1_std,Motor electrodynamic force_phase1_min,Motor electrodynamic force_phase1_max,Motor electrodynamic force_phase1_range,Motor electrodynamic force_phase2_mean,Motor electrodynamic force_phase2_std,Motor electrodynamic force_phase2_min,Motor electrodynamic force_phase2_max,Motor electrodynamic force_phase2_range,Motor electrodynamic force_phase3_mean,Motor electrodynamic force_phase3_std,Motor electrodynamic force_phase3_min,Motor electrodynamic force_phase3_max,Motor electrodynamic force_phase3_range,Motor electrodynamic force_phase4_mean,Motor electrodynamic force_phase4_std,Motor electrodynamic force_phase4_min,Motor electrodynamic force_phase4_max,Motor electrodynamic force_phase4_range,Motor electrodynamic force_phase5_mean,Motor electrodynamic force_phase5_std,Motor electrodynamic force_phase5_min,Motor electrodynamic force_phase5_max,Motor electrodynamic force_phase5_range,Door leaf position_phase1_mean,Door leaf position_phase1_std,Door leaf position_phase1_min,Door leaf position_phase1_max,Door leaf position_phase1_range,Door leaf position_phase2_mean,Door leaf position_phase2_std,Door leaf position_phase2_min,Door leaf position_phase2_max,Door leaf position_phase2_range,Door leaf position_phase3_mean,Door leaf position_phase3_std,Door leaf position_phase3_min,Door leaf position_phase3_max,Door leaf position_phase3_range,Door leaf position_phase4_mean,Door leaf position_phase4_std,Door leaf position_phase4_min,Door leaf position_phase4_max,Door leaf position_phase4_range,Door leaf position_phase5_mean,Door leaf position_phase5_std,Door leaf position_phase5_min,Door leaf position_phase5_max,Door leaf position_phase5_range
0,537.631579,426.440187,115.0,1432.0,1317.0,269.236842,19.330143,250.0,335.0,85.0,180.684211,85.979834,0.0,265.0,265.0,154.842105,151.433766,0.0,394.0,394.0,987.184211,806.85187,226.0,2208.0,1982.0,4565.789474,1704.989013,400.0,6200.0,5800.0,5421.052632,40.768246,5400.0,5500.0,100.0,4978.947368,554.00831,3600.0,5400.0,1800.0,2539.473684,413.928212,2000.0,3600.0,1600.0,3236.842105,836.163247,2400.0,5200.0,2800.0,945.736842,465.75178,53.0,1307.0,1254.0,1319.078947,5.367453,1300.0,1327.0,27.0,1245.421053,103.540897,968.0,1331.0,363.0,606.526316,147.891272,460.0,968.0,508.0,358.815789,231.841241,0.0,557.0,557.0,646.052632,47.771907,555.0,700.0,145.0,451.973684,61.196614,349.0,555.0,206.0,247.0,58.984387,152.0,349.0,197.0,99.078947,26

## Dynamic Features

Abnormal resistance may alter not only the absolute sensor values but
also how quickly those values change during an operation.

We therefore construct features based on first differences and signal
integrals.

These features capture aspects of signal shape that simple mean/max
statistics cannot represent.

In [13]:
# Calculate first-difference statistics for a signal.


def derivative_statistics(cycle, signal, prefix):

    values = pd.to_numeric(cycle[signal], errors="coerce").to_numpy(dtype=float)

    diff = np.diff(values)

    return {
        f"{prefix}_diff_mean": np.mean(diff),
        f"{prefix}_diff_std": np.std(diff),
        f"{prefix}_diff_min": np.min(diff),
        f"{prefix}_diff_max": np.max(diff),
        f"{prefix}_diff_abs_mean": np.mean(np.abs(diff)),
        f"{prefix}_diff_abs_max": np.max(np.abs(diff)),
    }

In [14]:
# Calculate how much the door position changes between consecutive samples.


def movement_statistics(cycle):

    position = pd.to_numeric(cycle["Door leaf position"], errors="coerce").to_numpy(dtype=float)

    movement = np.diff(position)
    abs_movement = np.abs(movement)

    return {
        "position_abs_diff_mean": np.mean(abs_movement),
        "position_abs_diff_std": np.std(abs_movement),
        "position_abs_diff_max": np.max(abs_movement),
        "position_zero_change_fraction": np.mean(movement == 0),
    }

In [15]:
# Calculate the approximate accumulated absolute motor effort over a cycle.


def integral_features(cycle):

    current = pd.to_numeric(cycle["Motor current(mA)"], errors="coerce").to_numpy(dtype=float)
    voltage = pd.to_numeric(cycle["Motor Voltage(10mV)"], errors="coerce").to_numpy(dtype=float)

    force = pd.to_numeric(cycle["Motor electrodynamic force"], errors="coerce").to_numpy(dtype=float)

    return {"current_abs_integral": np.sum(np.abs(current)), "voltage_abs_integral": np.sum(np.abs(voltage)), "force_abs_integral": np.sum(np.abs(force))}

## Timing and Operation Context

We retain cycle duration and operation type.

Open and Close operations have different physical behaviour, so the model
should be allowed to distinguish between them.

In [16]:
# Calculate timing information for one cycle.


def timing_features(cycle):

    duration = (cycle["timestamp"].iloc[-1] - cycle["timestamp"].iloc[0]).total_seconds()

    return {"duration_seconds": duration, "n_rows": len(cycle)}

In [17]:
# Encode Close as 1 and Open as 0 so operation can be used as a model input.


def operation_features(operation):

    return {"is_close": 1 if operation == "Close" else 0}

## Build Cycle-Level Feature Matrix

Each labelled door operation will now be converted into one row of
numerical features.

The original `operation` and `status` values are retained as metadata,
while `is_close` provides an encoded operation feature for later modelling.

In [18]:
# Build the full engineered feature table for all training cycles.

feature_rows = []

for _, segment in segments.iterrows():
    cycle = extract_cycle(train, segment)

    features = {}

    # Whole-cycle statistics
    for signal in SIGNALS:
        features.update(basic_statistics(cycle[signal], signal))

    # Phase-based statistics
    for signal in SIGNALS:
        for phase_number, (phase_start, phase_end) in enumerate(PHASES, start=1):
            features.update(phase_statistics(cycle, signal, phase_start, phase_end, phase_number))

    # Dynamic features
    features.update(derivative_statistics(cycle, "Door leaf position", "position"))

    features.update(derivative_statistics(cycle, "Motor current(mA)", "current"))

    features.update(movement_statistics(cycle))

    # Integral-like features
    features.update(integral_features(cycle))

    # Timing
    features.update(timing_features(cycle))

    # Operation
    features.update(operation_features(segment["operation"]))

    # Metadata / target
    features["segment_id"] = segment["segment_id"]
    features["operation"] = segment["operation"]
    features["status"] = segment["status"]

    feature_rows.append(features)

cycle_features = pd.DataFrame(feature_rows)

print("Feature table shape:", cycle_features.shape)

display(cycle_features.head())

Feature table shape: (110, 145)


,Motor current(mA)_mean,Motor current(mA)_std,Motor current(mA)_min,Motor current(mA)_max,Motor current(mA)_range,Motor Voltage(10mV)_mean,Motor Voltage(10mV)_std,Motor Voltage(10mV)_min,Motor Voltage(10mV)_max,Motor Voltage(10mV)_range,Motor electrodynamic force_mean,Motor electrodynamic force_std,Motor electrodynamic force_min,Motor electrodynamic force_max,Motor electrodynamic force_range,Door leaf position_mean,Door leaf position_std,Door leaf position_min,Door leaf position_max,Door leaf position_range,Motor current(mA)_phase1_mean,Motor current(mA)_phase1_std,Motor current(mA)_phase1_min,Motor current(mA)_phase1_max,Motor current(mA)_phase1_range,Motor current(mA)_phase2_mean,Motor current(mA)_phase2_std,Motor current(mA)_phase2_min,Motor current(mA)_phase2_max,Motor current(mA)_phase2_range,Motor current(mA)_phase3_mean,Motor current(mA)_phase3_std,Motor current(mA)_phase3_min,Motor current(mA)_phase3_max,Motor current(mA)_phase3_range,Motor current(mA)_phase4_mean,Motor current(mA)_phase4_std,Motor current(mA)_phase4_min,Motor current(mA)_phase4_max,Motor current(mA)_phase4_range,Motor current(mA)_phase5_mean,Motor current(mA)_phase5_std,Motor current(mA)_phase5_min,Motor current(mA)_phase5_max,Motor current(mA)_phase5_range,Motor Voltage(10mV)_phase1_mean,Motor Voltage(10mV)_phase1_std,Motor Voltage(10mV)_phase1_min,Motor Voltage(10mV)_phase1_max,Motor Voltage(10mV)_phase1_range,Motor Voltage(10mV)_phase2_mean,Motor Voltage(10mV)_phase2_std,Motor Voltage(10mV)_phase2_min,Motor Voltage(10mV)_phase2_max,Motor Voltage(10mV)_phase2_range,Motor Voltage(10mV)_phase3_mean,Motor Voltage(10mV)_phase3_std,Motor Voltage(10mV)_phase3_min,Motor Voltage(10mV)_phase3_max,Motor Voltage(10mV)_phase3_range,Motor Voltage(10mV)_phase4_mean,Motor Voltage(10mV)_phase4_std,Motor Voltage(10mV)_phase4_min,Motor Voltage(10mV)_phase4_max,Motor Voltage(10mV)_phase4_range,Motor Voltage(10mV)_phase5_mean,Motor Voltage(10mV)_phase5_std,Motor Voltage(10mV)_phase5_min,Motor Voltage(10mV)_phase5_max,Motor Voltage(10mV)_phase5_range,Motor electrodynamic force_phase1_mean,Motor electrodynamic force_phase1_std,Motor electrodynamic force_phase1_min,Motor electrodynamic force_phase1_max,Motor electrodynamic force_phase1_range,Motor electrodynamic force_phase2_mean,Motor electrodynamic force_phase2_std,Motor electrodynamic force_phase2_min,Motor electrodynamic force_phase2_max,Motor electrodynamic force_phase2_range,Motor electrodynamic force_phase3_mean,Motor electrodynamic force_phase3_std,Motor electrodynamic force_phase3_min,Motor electrodynamic force_phase3_max,Motor electrodynamic force_phase3_range,Motor electrodynamic force_phase4_mean,Motor electrodynamic force_phase4_std,Motor electrodynamic force_phase4_min,Motor electrodynamic force_phase4_max,Motor electrodynamic force_phase4_range,Motor electrodynamic force_phase5_mean,Motor electrodynamic force_phase5_std,Motor electrodynamic force_phase5_min,Motor electrodynamic force_phase5_max,Motor electrodynamic force_phase5_range,Door leaf position_phase1_mean,Door leaf position_phase1_std,Door leaf position_phase1_min,Door leaf position_phase1_max,Door leaf position_phase1_range,Door leaf position_phase2_mean,Door leaf position_phase2_std,Door leaf position_phase2_min,Door leaf position_phase2_max,Door leaf position_phase2_range,Door leaf position_phase3_mean,Door leaf position_phase3_std,Door leaf position_phase3_min,Door leaf position_phase3_max,Door leaf position_phase3_range,Door leaf position_phase4_mean,Door leaf position_phase4_std,Door leaf position_phase4_min,Door leaf position_phase4_max,Door leaf position_phase4_range,Door leaf position_phase5_mean,Door leaf position_phase5_std,Door leaf position_phase5_min,Door leaf position_phase5_max,Door leaf position_phase5_range,position_diff_mean,position_diff_std,position_diff_min,position_diff_max,position_diff_abs_mean,position_diff_abs_max,current_diff_mean,current_diff_std,current_diff_min,current_diff_max,current_diff_abs_mean,current_diff_abs_m

## Feature Quality Checks

Before classification, we verify that the generated feature matrix does
not contain invalid values and remove features that carry no variation.

In [19]:
# Separate metadata/target columns from numerical model features.

metadata_columns = ["segment_id", "operation", "status"]

numeric_features = [column for column in cycle_features.columns if column not in metadata_columns]

feature_matrix = cycle_features[numeric_features].copy()

print("Number of numerical features:", len(numeric_features))

Number of numerical features: 142


In [20]:
# Check for missing and infinite values in the feature matrix.

print("NaN values:", feature_matrix.isna().sum().sum())

print("Infinite values:", np.isinf(feature_matrix.to_numpy()).sum())

NaN values: 0
Infinite values: 0


In [21]:
# Identify features with no variation across the 110 training cycles.

constant_features = [column for column in feature_matrix.columns if feature_matrix[column].nunique() <= 1]

print("Constant features:", len(constant_features))
print(constant_features)

Constant features: 2
['Motor electrodynamic force_min', 'Motor electrodynamic force_phase5_min']


In [22]:
# Remove constant features because they cannot distinguish between classes.

if constant_features:
    cycle_features = cycle_features.drop(columns=constant_features)

numeric_features = [column for column in cycle_features.columns if column not in metadata_columns]

feature_matrix = cycle_features[numeric_features].copy()

print("Remaining numerical features:", len(numeric_features))

Remaining numerical features: 140


In [23]:
# Summarise the range and spread of each numerical feature.

feature_summary = feature_matrix.describe().T

display(feature_summary[["mean", "std", "min", "max"]])

,mean,std,min,max
Motor current(mA)_mean,587.223697,139.758131,418.596774,972.812500
Motor current(mA)_std,623.324956,127.206878,427.324029,872.394359
Motor current(mA)_min,15.336364,32.479571,0.000000,112.000000
Motor current(mA)_max,2305.563636,204.494477,1993.000000,2560.000000
Motor current(mA)_range,2290.227273,222.366491,1881.000000,2554.000000
...,...,...,...,...
voltage_abs_integral,799427.272727,37353.417266,762600.000000,911500.000000
force_abs_integral,166475.645455,1714.776154,163363.000000,169777.000000
duration_seconds,3.259273,0.442925,2.720000,3.780000
n_rows,163.963636,22.146272,137.000000,190.000000


In [24]:
# Find highly correlated pairs among the engineered numerical features.

correlation = feature_matrix.corr().abs()
upper = correlation.where(np.triu(np.ones(correlation.shape), k=1).astype(bool))
high_correlation_pairs = upper.stack().sort_values(ascending=False)
display(high_correlation_pairs.head(30))

position_diff_std                      position_abs_diff_std                      1.000000
position_diff_abs_mean                 position_abs_diff_mean                     1.000000
duration_seconds                       n_rows                                     1.000000
Motor Voltage(10mV)_min                Motor Voltage(10mV)_phase1_min             1.000000
current_diff_max                       current_diff_abs_max                       1.000000
Motor electrodynamic force_max         Motor electrodynamic force_range           1.000000
Motor electrodynamic force_phase5_max  Motor electrodynamic force_phase5_range    1.000000
position_diff_abs_max                  position_abs_diff_max                      1.000000
Door leaf position_phase5_mean         Door leaf position_phase5_max              0.999993
Door leaf position_phase5_min          is_close                                   0.999990
Door leaf position_phase5_mean         Door leaf position_phase5_min              0.999988

In [25]:
# Compare the average value of each engineered feature between the two classes.

class_means = cycle_features.groupby("status")[numeric_features].mean().T
display(class_means)

status,Abnormal resistance,Normal
Motor current(mA)_mean,719.838916,537.492990
Motor current(mA)_std,611.919813,627.601885
Motor current(mA)_min,50.833333,2.025000
Motor current(mA)_max,2255.100000,2324.487500
Motor current(mA)_range,2204.266667,2322.462500
...,...,...
voltage_abs_integral,853606.666667,779110.000000
force_abs_integral,164970.166667,167040.200000
duration_seconds,3.270667,3.255000
n_rows,164.533333,163.750000


In [27]:
# Calculate a simple mean-difference ratio to identify potentially informative features.

normal_mean = class_means["Normal"]
abnormal_mean = class_means["Abnormal resistance"]

feature_difference = pd.DataFrame({"normal_mean": normal_mean, "abnormal_mean": abnormal_mean})

feature_difference["absolute_difference"] = (feature_difference["abnormal_mean"] - feature_difference["normal_mean"]).abs()

feature_difference["relative_difference"] = feature_difference["absolute_difference"] / (np.abs(feature_difference["normal_mean"]) + 1e-9)

display(feature_difference.sort_values("relative_difference", ascending=False).head(30))

,normal_mean,abnormal_mean,absolute_difference,relative_difference
Door leaf position_min,0.000000,0.333333,0.333333,3.333333e+08
Motor current(mA)_phase4_min,2.025000,115.300000,113.275000,5.593827e+01
Motor current(mA)_min,2.025000,50.833333,48.808333,2.410288e+01
Motor current(mA)_phase3_min,24.037500,193.500000,169.462500,7.049922e+00
Motor Voltage(10mV)_phase2_range,117.500000,793.333333,675.833333,5.751773e+00
Motor Voltage(10mV)_phase2_std,39.269157,248.937435,209.668278,5.339261e+00
Motor current(mA)_phase2_std,47.178141,169.267487,122.089345,2.587837e+00
Motor current(mA)_phase2_range,202.150000,587.300000,385.150000,1.905268e+00
Motor current(mA)_phase3_mean,192.015887,537.636126,345.620239,1.799956e+00
Motor current(mA)_phase3_max,284.350000,769.666667,485.316667,1.706758e+00


In [28]:
# Save the engineered cycle-level dataset for use by the classification notebook.

OUTPUT_PATH = Path("door_cycle_features.csv")

cycle_features.to_csv(OUTPUT_PATH, index=False)

print("Saved:", OUTPUT_PATH.resolve())
print("Final shape:", cycle_features.shape)

Saved: /Users/yeo/Documents/Door/NOTEBOOKS/door_cycle_features.csv
Final shape: (110, 143)
